## MODELO BASE (BASELINE)

### By:
Cristian David Ceballos Velez

### Date:
2026-08-25

### Description:
Creación de un modelo base (baseline) para el dataset `corazon.csv`, como referencia para
comparar contra modelos de machine learning más complejos en etapas posteriores. Se evalúan
dos enfoques: un **modelo Dummy** (referencia mínima exigida por scikit-learn) y un
**modelo heurístico** basado en reglas clínicas conocidas para enfermedad cardíaca.

Estructura basada en: [Modelo Base - Jose R. Zapata](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/5-baseline_model/)

> **Nota:** las reglas del modelo heurístico (sección 6.2) están basadas en conocimiento clínico
> general del dataset de Cleveland (origen común de este tipo de datasets). **Ajusta los umbrales
> y variables según los resultados reales de tu análisis bivariable/multivariable (`3-analysis`).**

## 📚 1. Importar librerías

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    classification_report,
)
from sklearn.model_selection import (
    KFold,
    ShuffleSplit,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

print("Pandas version:", pd.__version__)

## 💾 2. Cargar datos

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

df = pd.read_parquet(DATA_DIR / "02_intermediate" / "corazon_type_fixed.parquet", engine="pyarrow")

TARGET = "disease"
df[TARGET] = df[TARGET].astype(int)

df.info()

## 🔎 3. Selección de la variable más importante

Antes de definir las reglas del modelo heurístico, se cuantifica la asociación de cada
variable con el target: correlación biserial-puntual para numéricas, y Chi² (V de Cramér)
para categóricas. Esto complementa (o reemplaza) los resultados del análisis bivariable
del notebook de EDA.

In [ ]:
cols_numeric_all = list(df.select_dtypes(include=["number"]).columns)
if TARGET in cols_numeric_all:
    cols_numeric_all.remove(TARGET)

cols_categoric_all = list(df.select_dtypes(include=["category", "object", "boolean"]).columns)
if TARGET in cols_categoric_all:
    cols_categoric_all.remove(TARGET)

# Asociación de variables numéricas con el target (correlación punto-biserial)
resultados_numericas = []
for col in cols_numeric_all:
    datos = df[[col, TARGET]].dropna()
    corr, p_valor = stats.pointbiserialr(datos[TARGET], datos[col])
    resultados_numericas.append({"variable": col, "correlacion": corr, "p_valor": p_valor})

df_num = pd.DataFrame(resultados_numericas).sort_values("correlacion", key=abs, ascending=False)
df_num

In [ ]:
# Asociación de variables categóricas con el target (Chi² y V de Cramér)
resultados_categoricas = []
for col in cols_categoric_all:
    tabla = pd.crosstab(df[col], df[TARGET])
    chi2, p_valor, dof, _ = stats.chi2_contingency(tabla)
    n = tabla.sum().sum()
    v_cramer = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
    resultados_categoricas.append({"variable": col, "v_cramer": v_cramer, "p_valor": p_valor})

df_cat = pd.DataFrame(resultados_categoricas).sort_values("v_cramer", ascending=False)
df_cat

**Justificación de la variable más importante:** revisa la tabla anterior — la variable con
mayor correlación (numéricas) o V de Cramér (categóricas) y p-valor significativo (< 0.05) es la
que más información aporta sobre `disease`. En datasets clínicos de este tipo, `thal`, `ca` y
`chest_pain` suelen ser las más fuertes. **Reemplaza el texto con tu hallazgo real** una vez
ejecutes las celdas anteriores con tus datos.

## 👷 4. Preparación de datos para el modelo heurístico

Para el modelo heurístico solo se necesitan las variables identificadas como más relevantes en la sección 3. Ajusta `selected_features` según tus resultados.

In [ ]:
selected_features = ["chest_pain", "thal", "ca", TARGET]  # <-- AJUSTAR según sección 3

heart_baseline = df[selected_features].copy()
heart_baseline.isna().sum()

## ✂️ 5. Train / Test split

In [ ]:
X_features = heart_baseline.drop(columns=[TARGET])
y_target = heart_baseline[TARGET]

x_train, x_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, stratify=y_target, random_state=42
)

print("Train:", x_train.shape, y_train.shape)
print("Test:", x_test.shape, y_test.shape)

## 🔧 6. Modelos base

### 6.1 Preprocesador

Para el modelo heurístico las columnas categóricas pasan sin transformar (`passthrough`), solo
se imputan valores faltantes en `ca` (numérica).

In [ ]:
cols_categoric = ["chest_pain", "thal"]  # <-- AJUSTAR
cols_numeric = ["ca"]  # <-- AJUSTAR

numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categoric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])

preprocessor = ColumnTransformer(
    transformers=[
        ("categoric", categoric_pipe, cols_categoric),
        ("numeric", numeric_pipe, cols_numeric),
    ]
)
preprocessor

### 6.2 Modelo heurístico

Regla clínica de referencia (**ajustar según hallazgos reales de tu EDA**):
- Si `chest_pain` = "asymptomatic", o
- `thal` en {"fixed", "reversable"} (defecto de perfusión), o
- `ca` > 0 (vasos principales coloreados por fluoroscopia)

→ se predice `disease` = 1, de lo contrario 0.

In [ ]:
class HeuristicModel(BaseEstimator, ClassifierMixin):
    """Modelo heurístico basado en reglas clínicas para el dataset de corazón."""

    def fit(self, X, y=None):
        if y is not None:
            self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        chest_pain_riesgo = "asymptomatic"
        thal_riesgo = {"fixed", "reversable"}
        ca_umbral = 0

        predictions = []
        for row in X:
            chest_pain, thal, ca = row[0], row[1], row[2]
            if (chest_pain == chest_pain_riesgo) or (thal in thal_riesgo) or (ca > ca_umbral):
                predictions.append(1)
            else:
                predictions.append(0)
        return np.array(predictions)

### 6.3 Modelo Dummy (referencia mínima)

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)

## 📊 7. Validación cruzada — comparación Dummy vs. Heurístico

In [ ]:
scoring_metrics = ["accuracy", "f1", "precision", "recall"]
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

modelos = {
    "Dummy": dummy_model,
    "Heuristico": HeuristicModel(),
}

resultados_cv = {}

for nombre_modelo, modelo in modelos.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", modelo)])
    resultados_cv[nombre_modelo] = {}
    for metric in scoring_metrics:
        scores = cross_val_score(pipe, x_train, y_train, cv=kfold, scoring=metric)
        resultados_cv[nombre_modelo][metric] = scores

for nombre_modelo, metricas in resultados_cv.items():
    print(f"\n--- {nombre_modelo} ---")
    for metric, scores in metricas.items():
        print(f"{metric}: media={scores.mean():.4f}, std={scores.std():.4f}")

In [ ]:
# Boxplot comparativo de las métricas de validación cruzada
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (nombre_modelo, metricas) in zip(axes, resultados_cv.items()):
    pd.DataFrame(metricas).plot.box(ax=ax, title=f"Cross Validation - {nombre_modelo}")
    ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

## 🎯 8. Evaluación final en test set (modelo heurístico)

In [ ]:
model_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", HeuristicModel())])
model_pipe.fit(x_train, y_train)

y_pred = model_pipe.predict(x_test)
print(classification_report(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred);

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test, y_pred);

### 8.1 ¿Por qué Recall como métrica principal?

En un problema de diagnóstico de enfermedad cardíaca, el error más costoso es un
**Falso Negativo** (decirle a un paciente enfermo que está sano) — puede significar no recibir
tratamiento a tiempo. Por eso, entre las métricas evaluadas, **Recall** (sensibilidad) es la más
relevante para este problema: mide qué proporción de los pacientes realmente enfermos el modelo
logra identificar correctamente.

## 📈 9. Learning Curve (basada en Recall)

In [ ]:
model = HeuristicModel()
model_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

common_params = {
    "X": x_train,
    "y": y_train,
    "train_sizes": np.linspace(0.1, 1.0, 5),
    "cv": ShuffleSplit(n_splits=50, test_size=0.2, random_state=123),
    "n_jobs": -1,
    "return_times": True,
}

scoring_metric = "recall"

train_sizes, train_scores, test_scores, fit_times, score_times = learning_curve(
    model_pipe, **common_params, scoring=scoring_metric
)

train_mean, train_std = np.mean(train_scores, axis=1), np.std(train_scores, axis=1)
test_mean, test_std = np.mean(test_scores, axis=1), np.std(test_scores, axis=1)
fit_times_mean, fit_times_std = np.mean(fit_times, axis=1), np.std(fit_times, axis=1)
score_times_mean, score_times_std = np.mean(score_times, axis=1), np.std(score_times, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_mean, "o-", label="Training score")
ax.plot(train_sizes, test_mean, "o-", color="orange", label="Cross-validation score")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.3)
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.3, color="orange")
ax.set_title(f"Learning Curve - {model.__class__.__name__} (Recall)")
ax.set_xlabel("Ejemplos de entrenamiento")
ax.set_ylabel(scoring_metric)
ax.legend(loc="best")
plt.show()

print("Training Sizes:", train_sizes)
print("Training Scores Mean:", train_mean)
print("Training Scores Std:", train_std)
print("Test Scores Mean:", test_mean)
print("Test Scores Std:", test_std)

## ⚙️ 10. Escalabilidad (tiempo de entrenamiento y score)

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 10), sharex=True)

ax[0].plot(train_sizes, fit_times_mean, "o-")
ax[0].fill_between(train_sizes, fit_times_mean - fit_times_std, fit_times_mean + fit_times_std, alpha=0.3)
ax[0].set_ylabel("Fit time (s)")
ax[0].set_title(f"Escalabilidad - {model.__class__.__name__}")

ax[1].plot(train_sizes, score_times_mean, "o-")
ax[1].fill_between(train_sizes, score_times_mean - score_times_std, score_times_mean + score_times_std, alpha=0.3)
ax[1].set_ylabel("Score time (s)")
ax[1].set_xlabel("Número de ejemplos de entrenamiento")

plt.show()

print("Fit Times Mean:", fit_times_mean)
print("Fit Times Std:", fit_times_std)
print("Score Times Mean:", score_times_mean)
print("Score Times Std:", score_times_std)

## 🔬 11. Interpretación de la Learning Curve

_Completar con base en las gráficas obtenidas:_

- **Brecha train/test**: si el score de entrenamiento es mucho mayor al de validación cruzada,
  hay señal de sobreajuste (poco esperable en un modelo heurístico simple, pero verificar).
- **Tendencia con más datos**: si el score de validación mejora al aumentar los datos de
  entrenamiento, sugiere que el modelo podría beneficiarse de más datos; si se estabiliza rápido
  (como es típico en un modelo de reglas fijas), significa que el modelo ya "convergió" y más
  datos no lo van a mejorar — hace falta un modelo con capacidad de aprendizaje real.
- **Escalabilidad**: el modelo heurístico debería tener tiempos de fit/score muy bajos y estables
  (no "aprende" parámetros), lo cual es esperado y no es una ventaja comparativa real frente a
  modelos de ML (que sí se benefician de más datos y cómputo).

## 📈 12. Análisis de resultados

_Completar con los valores numéricos reales obtenidos:_

### Resultados de validación cruzada
- **Accuracy**: media = _, std = _
- **F1**: media = _, std = _
- **Precision**: media = _, std = _
- **Recall**: media = _, std = _

### Comparación Dummy vs. Heurístico
- ¿El modelo heurístico supera al Dummy en Recall? (debería, si las reglas capturan señal real)
- ¿Qué tan grande es la diferencia? Esto define qué tan útil es el conocimiento de dominio
  incorporado en las reglas frente a no usar información alguna (Dummy).

## 📝 13. Conclusiones

- **Modelo base seleccionado:** modelo heurístico (basado en `chest_pain`, `thal`, `ca`)
- **Métrica principal:** Recall, por el costo alto de los Falsos Negativos en diagnóstico clínico
- **Desempeño general:** (completar con los números obtenidos)
- **Generalización:** (completar según la brecha train/test de la learning curve)
- **Utilidad como referencia:** este modelo baseline marca el piso mínimo de desempeño (Recall,
  Precision, F1) que cualquier modelo de ML entrenado en la siguiente etapa debe superar para
  justificar su mayor complejidad.

## 🧑‍🔬 14. Recomendaciones

1. **Usar este modelo como piso de comparación**: cualquier modelo de ML debe superar,
   como mínimo, el Recall obtenido por el modelo heurístico en el test set.
2. **No usar el modelo heurístico en producción**: sus reglas fijas no capturan interacciones
   complejas entre variables ni se ajustan a nuevos datos.
3. **Monitorear la métrica elegida (Recall)** de forma consistente en las siguientes etapas,
   para mantener comparabilidad entre modelos.

## 💡 15. Propuestas e ideas

1. **Modelos más complejos**: evaluar Regresión Logística, Random Forest, Gradient Boosting
   (ej. XGBoost) — pueden capturar interacciones no lineales entre variables clínicas.
2. **Usar todas las variables**: el modelo heurístico usa solo 3 columnas; los modelos de ML
   de la siguiente etapa deben aprovechar el pipeline completo de `4-feat_eng` (todas las
   variables, con escalado y encoding).
3. **Ajuste de hiperparámetros**: Grid Search / Random Search una vez seleccionado el modelo.
4. **Validación cruzada estratificada**: mantener `stratify` en el split y considerar
   `StratifiedKFold` en vez de `KFold` simple, dado el (leve) desbalance de clases del target.
5. **Costo de los errores**: considerar ajustar el umbral de decisión (no solo 0.5) en modelos
   probabilísticos, dado que el Falso Negativo es más costoso que el Falso Positivo en este
   problema clínico.